# Assignment 1 — Machine Learning (26DS601)
### Dataset: Pima Indians Diabetes Dataset (UCI / public domain)

This notebook covers all required tasks:
1. Data visualization (boxplot, scatter, histogram, pairplot)
2. Correlation heatmap and feature selection discussion
3. Data quality report (missing values, cardinality, feature types)
4. Data scaling — rationale
5. Class distribution plot and imbalance discussion
6. Missing value handling
7. Class imbalance handling + classification with KNN and Naive Bayes


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report,
                              roc_auc_score, roc_curve)
from imblearn.over_sampling import SMOTE

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
RANDOM_STATE = 42


## 1. Load the dataset
Fetched directly from a public raw-CSV source (Pima Indians Diabetes Dataset).

In [ ]:
columns = ["Pregnancies","Glucose","BloodPressure","SkinThickness","Insulin",
           "BMI","DiabetesPedigreeFunction","Age","Outcome"]

url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
df = pd.read_csv(url, names=columns)

print("Shape:", df.shape)
df.head()


## 2. Data quality report
Missing values, cardinality, and feature types.

Note: in this dataset, missing values are encoded as `0` in physiologically
implausible columns (Glucose, BloodPressure, SkinThickness, Insulin, BMI — a
living person cannot have 0 for these), rather than as `NaN`. We first
convert those zeros to NaN so pandas' missing-value tools report them correctly.

In [ ]:
zero_as_missing_cols = ["Glucose","BloodPressure","SkinThickness","Insulin","BMI"]
df[zero_as_missing_cols] = df[zero_as_missing_cols].replace(0, np.nan)

print("=== Data types ===")
print(df.dtypes)

print("\n=== Missing value count & percentage ===")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
quality_report = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct,
                                "cardinality": df.nunique(), "dtype": df.dtypes})
quality_report


**Observations:**
- All 9 columns are numeric (int64/float64) — no categorical/text features in this dataset.
- `Insulin` and `SkinThickness` have the highest missing percentage (~48.7% and ~29.6% respectively).
- `Glucose`, `BloodPressure`, `BMI` have a small number of missing values.
- Cardinality is high for continuous features (Glucose, BMI, etc.) and low for `Outcome` (2 — it's the binary target) and `Pregnancies` (bounded integer range).

## 3. Data visualization
### 3.1 Boxplots — outlier and spread inspection per feature

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.flatten(), columns[:-1]):
    sns.boxplot(y=df[col], ax=ax, color="#7F77DD")
    ax.set_title(col)
plt.tight_layout()
plt.show()


### 3.2 Histograms — distribution shape per feature

In [ ]:
df[columns[:-1]].hist(bins=20, figsize=(16, 10), color="#1D9E75", edgecolor="black")
plt.tight_layout()
plt.show()


### 3.3 Scatter plot — feature relationship (Glucose vs BMI, colored by Outcome)

In [ ]:
plt.figure(figsize=(7,5))
sns.scatterplot(data=df, x="Glucose", y="BMI", hue="Outcome", palette=["#378ADD","#D85A30"])
plt.title("Glucose vs BMI by diabetes outcome")
plt.show()


### 3.4 Pairplot — pairwise relationships across key features

In [ ]:
sns.pairplot(df, vars=["Glucose","BMI","Age","Insulin"], hue="Outcome",
             palette=["#378ADD","#D85A30"], diag_kind="hist")
plt.show()


## 4. Correlation heatmap and feature selection

We compute the correlation matrix (Pearson) across all features and the target
(`Outcome`) to identify which attributes carry the most linear signal for
classification.

In [ ]:
plt.figure(figsize=(9,7))
corr = df.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation heatmap")
plt.show()

print("Correlation with target (Outcome), sorted:")
print(corr["Outcome"].sort_values(ascending=False))


**Feature selection discussion:**
`Glucose` shows the strongest correlation with `Outcome`, followed by `BMI`,
`Age`, and `DiabetesPedigreeFunction`. `SkinThickness` and `BloodPressure`
show comparatively weak correlation with the target. Given the dataset only
has 8 predictor features (already a small set) and no pair of features shows
extreme multicollinearity (no |r| > 0.9 between predictors), we retain **all
8 features** for classification rather than dropping any — removing already-few
features risks losing useful non-linear signal that KNN/Naive Bayes could
still exploit, even where linear correlation is weak.

## 5. Data scaling — is it needed?

**Yes, scaling is required here**, for two specific reasons tied to the models used:
- **KNN** is distance-based (Euclidean distance by default). Features like
  `Insulin` (range ~0–850) and `DiabetesPedigreeFunction` (range ~0.08–2.4)
  are on wildly different scales — without scaling, `Insulin` would dominate
  the distance calculation and effectively drown out smaller-range features.
- **Gaussian Naive Bayes** assumes each feature is normally distributed;
  standardizing (mean 0, std 1) doesn't change the shape of the distribution,
  but keeps numerical values in a consistent, well-conditioned range for
  stable variance estimation.

We use `StandardScaler` (z-score standardization) rather than min-max scaling,
since several features have outliers (visible in the boxplots above) and
StandardScaler is less sensitive to those than min-max scaling.

## 6. Handle missing values
Impute using the median (robust to the outliers/skew seen in the boxplots and histograms above), grouped by class where possible to preserve class-conditional structure.

In [ ]:
for col in zero_as_missing_cols:
    df[col] = df.groupby("Outcome")[col].transform(lambda x: x.fillna(x.median()))

print("Remaining missing values after imputation:")
print(df.isnull().sum())


## 7. Class distribution and imbalance

In [ ]:
counts = df["Outcome"].value_counts()
pct = (counts / len(df) * 100).round(2)
print("Class counts:\n", counts)
print("\nClass percentage:\n", pct)

plt.figure(figsize=(5,4))
sns.countplot(x="Outcome", data=df, palette=["#378ADD","#D85A30"])
plt.title("Class distribution: Outcome")
plt.xticks([0,1], ["No diabetes (0)", "Diabetes (1)"])
plt.show()


**Class imbalance discussion:** the dataset shows a moderate imbalance —
roughly 65% negative (no diabetes) vs 35% positive (diabetes). This is not
severe enough to require heavy techniques (e.g. extreme undersampling), but
is enough to bias a classifier toward the majority class if left unaddressed,
particularly hurting recall on the positive (diabetic) class — which is the
class that matters most in a medical screening context. We address this
with SMOTE (Synthetic Minority Over-sampling) on the training set only.

## 8. Classification — KNN and Naive Bayes
### 8.1 Train/test split, scaling, and SMOTE (train set only)

In [ ]:
X = df.drop(columns=["Outcome"])
y = df["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Before SMOTE:", y_train.value_counts().to_dict())
smote = SMOTE(random_state=RANDOM_STATE)
X_train_bal, y_train_bal = smote.fit_resample(X_train_scaled, y_train)
print("After SMOTE:", pd.Series(y_train_bal).value_counts().to_dict())


### 8.2 Train and evaluate KNN

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_bal, y_train_bal)
knn_pred = knn.predict(X_test_scaled)
knn_proba = knn.predict_proba(X_test_scaled)[:,1]

print("=== KNN classification report ===")
print(classification_report(y_test, knn_pred, target_names=["No diabetes","Diabetes"]))
print("ROC-AUC:", round(roc_auc_score(y_test, knn_proba), 3))

plt.figure(figsize=(4,4))
sns.heatmap(confusion_matrix(y_test, knn_pred), annot=True, fmt="d", cmap="Blues")
plt.title("KNN — confusion matrix")
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.show()


### 8.3 Train and evaluate Naive Bayes

In [ ]:
nb = GaussianNB()
nb.fit(X_train_bal, y_train_bal)
nb_pred = nb.predict(X_test_scaled)
nb_proba = nb.predict_proba(X_test_scaled)[:,1]

print("=== Naive Bayes classification report ===")
print(classification_report(y_test, nb_pred, target_names=["No diabetes","Diabetes"]))
print("ROC-AUC:", round(roc_auc_score(y_test, nb_proba), 3))

plt.figure(figsize=(4,4))
sns.heatmap(confusion_matrix(y_test, nb_pred), annot=True, fmt="d", cmap="Greens")
plt.title("Naive Bayes — confusion matrix")
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.show()


### 8.4 Model comparison

In [ ]:
results = pd.DataFrame({
    "Model": ["KNN", "Naive Bayes"],
    "Accuracy": [accuracy_score(y_test, knn_pred), accuracy_score(y_test, nb_pred)],
    "Precision": [precision_score(y_test, knn_pred), precision_score(y_test, nb_pred)],
    "Recall": [recall_score(y_test, knn_pred), recall_score(y_test, nb_pred)],
    "F1-score": [f1_score(y_test, knn_pred), f1_score(y_test, nb_pred)],
    "ROC-AUC": [roc_auc_score(y_test, knn_proba), roc_auc_score(y_test, nb_proba)],
}).round(3)
results


## 9. Conclusion

Both KNN and Naive Bayes were trained on the SMOTE-balanced, scaled, missing-value-imputed
Pima Indians Diabetes dataset. Fill in the actual numbers from the results table above
once you run this notebook, and briefly state which model performed better and why
(e.g. Naive Bayes's independence assumption vs KNN's sensitivity to local density) —
this discussion is expected in your write-up.